# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

**I used a line chart to compare population growth across different continents from 1970 to 2022. A line chart makes it easy to understand how the population has changed over the years. I also used the same starting value for all continents so that the comparison is fair. The chart helps us see which continents have grown faster and which have grown more slowly. It also shows the overall population growth trend over time.**

In [3]:
# Q1
# YOUR CODE HERE
# Q1: Population growth by continent since 1970

years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]

# Total population per continent
continent_pop = df.groupby("Continent")[
    [f"{year} Population" for year in years]
].sum()

# Normalize to 1970 = 100
growth_index = continent_pop.div(
    continent_pop["1970 Population"], axis=0
) * 100

# Prepare data for visualization
growth_long = growth_index.reset_index().melt(
    id_vars="Continent",
    var_name="Year",
    value_name="Growth Index"
)

growth_long["Year"] = growth_long["Year"].str[:4].astype(int)

# Interactive line chart
fig = px.line(
    growth_long,
    x="Year",
    y="Growth Index",
    color="Continent",
    markers=True,
    title="Population Growth by Continent (1970–2022)",
    labels={"Growth Index": "Population Index (1970 = 100)"}
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# Identify fastest-growing continent
print("Fastest-growing continent:")
print(growth_index["2022 Population"].idxmax())

Fastest-growing continent:
Africa


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
**I used a horizontal bar chart to show the 15 countries with the highest population increase between 1970 and 2022. This chart makes it easy to compare the population growth of different countries. The longer bars represent countries that added more people. I also used different colours for continents so that we can easily identify which countries belong to Africa. This helps us understand which countries contributed the most to global population growth.**

In [6]:
# Calculate population increase
df["Population Increase"] = df["2022 Population"] - df["1970 Population"]

# Top 15 countries
top15 = df.nlargest(15, "Population Increase")

# Bar chart
fig = px.bar(
    top15,
    x="Population Increase",
    y="Country",
    color="Continent",
    orientation="h",
    title="Top 15 Countries by Population Increase (1970–2022)"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    template="plotly_white"
)

fig.show()

# Count African countries
africa_count = (top15["Continent"] == "Africa").sum()
print("Number of African countries:", africa_count)

Number of African countries: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

**I used a scatter plot to compare the land area and population of different countries. Each dot represents a country, and different colours represent different continents. I used a logarithmic scale because the countries have very different population sizes and land areas. This makes the chart easier to read and compare.**

**The chart shows that countries with larger land areas do not always have larger populations. For example, India has a very high population compared to its land area, while Canada and Australia have large areas but smaller populations. This shows that land area alone does not determine a country's population.**

In [8]:
fig = px.scatter(
    df,
    x="Area",
    y="2022 Population",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    opacity=0.7,
    title="Land Area vs Population (2022)",
    labels={
        "Area": "Land Area (km²) - Log Scale",
        "2022 Population": "Population - Log Scale"
    }
)

fig.update_traces(marker=dict(size=8))

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=600
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
**I used a box plot to compare population density across different continents. This chart helps us understand how crowded countries are within each continent. The boxes show the middle range of population densities, while the points outside represent countries with unusually high or low density.**

**I used a logarithmic scale because population density varies a lot between countries. Some countries have very high population density, while others have much lower density. This chart makes it easier to compare the differences and understand how population density is spread across continents.**

In [9]:
# Q4
# YOUR CODE 
# Q4: Population density by continent

fig = px.box(
    df,
    x="Continent",
    y="Density",
    color="Continent",
    points="outliers",
    log_y=True,
    title="Population Density Across Continents",
    labels={
        "Density": "Population Density (people/km²)",
        "Continent": "Continent"
    }
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=600,
    showlegend=False
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

**I used a world map to show how the population changed in different countries between 2000 and 2022. The colours help us identify countries where the population increased or decreased. Green represents population growth, while red represents population decline.**

**I chose this chart because it makes it easy to understand where population changes happened around the world. We can also compare different regions and see which countries experienced higher growth. The map gives a clear picture of global population changes over these years.**

In [15]:
# Q5: Population growth from 2000 to 2022

df["Growth (%)"] = (
    (df["2022 Population"] - df["2000 Population"])
    / df["2000 Population"]
) * 100

fig = px.choropleth(
    df,
    locations="CCA3",
    color="Growth (%)",
    hover_name="Country",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title="Population Growth and Decline (2000–2022)"
)

fig.update_coloraxes(reversescale=True)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=600
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

**I used a sunburst chart to show how the world's population is distributed across continents and countries in 2022. The inner circle represents the continents, while the outer circle shows the countries within each continent.**

**The size of each section depends on its population. Larger sections represent countries or continents with more people. This chart makes it easy to understand how the world's population is divided and compare different regions. We can also see that Asia has the largest share of the global population.**

In [16]:
# Q6
# YOUR CODE HERE
# Q6: World population distribution in 2022

fig = px.sunburst(
    df,
    path=["Continent", "Country"],
    values="2022 Population",
    color="Continent",
    title="World Population Distribution by Continent and Country (2022)"
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=750
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

**I used a stacked area chart to show how the share of the world's population has changed between 1970 and 2022. I focused on Africa, Asia and Europe, while grouping the remaining continents as the rest of the world.**

**The chart shows that Africa's share has increased over time, while Europe's share has decreased. Asia continues to have the largest share of the world's population. I chose this chart because it makes it easy to compare the changes over time and see how the total population is divided between different regions.**

In [17]:
# Q7
# YOUR CODE HERE
# Q7: World population share over time

years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]

data = []

for year in years:
    population = df.groupby("Continent")[f"{year} Population"].sum()
    total = population.sum()

    for continent, value in population.items():
        region = continent if continent in ["Africa", "Asia", "Europe"] else "Rest of the world"

        data.append({
            "Year": year,
            "Region": region,
            "Population Share": value / total * 100
        })

share_df = pd.DataFrame(data)

share_df = share_df.groupby(
    ["Year", "Region"], as_index=False
)["Population Share"].sum()

fig = px.area(
    share_df,
    x="Year",
    y="Population Share",
    color="Region",
    title="World Population Share by Region (1970–2022)",
    labels={"Population Share": "Share of World Population (%)"},
    category_orders={
        "Region": ["Africa", "Asia", "Europe", "Rest of the world"]
    }
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=600,
    yaxis_range=[0, 100]
)

fig.show()